In [1]:
import os
import sys
import random
import numpy as np
import pandas as pd

import subprocess

try:
    import google.protobuf  # noqa: F401
    import google.protobuf as _gp
    from packaging import version

    if version.parse(_gp.__version__) >= version.parse("4.0.0"):
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "protobuf==3.20.3"]
        )
        import importlib

        importlib.invalidate_caches()
        for k in list(sys.modules.keys()):
            if k.startswith("google.protobuf"):
                del sys.modules[k]
except Exception as e:
    print("Warning: protobuf pin attempt failed:", repr(e))

import tensorflow as tf
import tensorflow.keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, LSTM, Conv1D

random.seed(7)
np.random.seed(7)
tf.random.set_seed(7)
try:
    tf.config.experimental.enable_op_determinism(True)
except Exception:
    pass

try:
    ncpu = os.cpu_count() or 4
    tf.config.threading.set_intra_op_parallelism_threads(min(8, ncpu))
    tf.config.threading.set_inter_op_parallelism_threads(min(2, ncpu))
except Exception:
    pass

print("TF version:", tf.__version__)



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 5.1 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 3.20.3 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 3.20.3 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.10.0 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.

TF version: 2.18.0


In [2]:
directory = "../input/ventilator-pressure-prediction"

train_dtypes = {
    "id": "int32",
    "breath_id": "int32",
    "R": "int16",
    "C": "int16",
    "time_step": "float32",
    "u_in": "float32",
    "u_out": "int8",
    "pressure": "float32",
}
test_dtypes = {
    "id": "int32",
    "breath_id": "int32",
    "R": "int16",
    "C": "int16",
    "time_step": "float32",
    "u_in": "float32",
    "u_out": "int8",
}

train = pd.read_csv(
    os.path.join(directory, "train.csv"),
    usecols=["id", "breath_id", "R", "C", "time_step", "u_in", "u_out", "pressure"],
    dtype=train_dtypes,
)
test = pd.read_csv(
    os.path.join(directory, "test.csv"),
    usecols=["id", "breath_id", "R", "C", "time_step", "u_in", "u_out"],
    dtype=test_dtypes,
)
sub = pd.read_csv(os.path.join(directory, "sample_submission.csv"))

print(train.shape, test.shape, sub.shape)



(5432400, 8) (603600, 7) (603600, 2)


In [3]:
features = ["R", "C", "time_step", "u_in", "u_out"]
_ = train[features].values  # original "sanity" access



In [4]:
sample_length = 80

X = train[features].to_numpy(dtype=np.float32, copy=False)
Y = train["pressure"].to_numpy(dtype=np.float32, copy=False)

n_train_steps = (len(train) // sample_length) * sample_length
X = X[:n_train_steps]
Y = Y[:n_train_steps]

inputs = X.reshape(-1, sample_length, len(features))
targets = Y.reshape(-1, sample_length, 1)

print("inputs shape:", inputs.shape, "targets shape:", targets.shape)



inputs shape: (67905, 80, 5) targets shape: (67905, 80, 1)


In [5]:
print("Train rows:", train.shape[0], "Test rows:", test.shape[0])




Train rows: 5432400 Test rows: 603600


In [6]:
def cnn_lstm_model():
    model = Sequential()
    model.add(Input(shape=(inputs.shape[1], inputs.shape[2])))
    model.add(
        Conv1D(
            filters=320, kernel_size=3, strides=1, padding="causal", activation="relu"
        )
    )
    model.add(LSTM(320, return_sequences=True, activation="tanh"))
    model.add(LSTM(320, return_sequences=True, activation="tanh"))
    model.add(Dense(160, activation="relu"))
    model.add(Dense(1, kernel_initializer="normal"))
    model.compile(
        loss="mae",
        optimizer=tf.keras.optimizers.Adam(),
        metrics=[tf.keras.metrics.RootMeanSquaredError()],
    )
    return model




In [7]:
BATCH_SIZE = 512
SHUFFLE_BUFFER = min(inputs.shape[0], 8192)

train_ds = (
    tf.data.Dataset.from_tensor_slices((inputs, targets))
    .shuffle(SHUFFLE_BUFFER, seed=7, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

model = cnn_lstm_model()
print(model.summary())

EPOCHS = int(os.environ.get("EPOCHS", "8"))
history = model.fit(train_ds, epochs=EPOCHS, verbose=2)



2026-01-21 10:29:02.065027: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1768991342.065485      11 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:c6:00.0, compute capability: 8.6


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 80, 320)        │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 80, 320)        │       820,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 80, 320)        │       820,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 80, 160)        │        51,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 80, 1)          │           161 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,697,601 (6.48 MB)

 Trainable params: 1,697,601 (6.48 MB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/8


I0000 00:00:1768991345.610201      70 cuda_dnn.cc:529] Loaded cuDNN version 90300


133/133 - 7s - 49ms/step - loss: 3.6252 - root_mean_squared_error: 6.9477
Epoch 2/8
133/133 - 3s - 26ms/step - loss: 1.0284 - root_mean_squared_error: 1.9342
Epoch 3/8
133/133 - 3s - 26ms/step - loss: 0.8363 - root_mean_squared_error: 1.4943
Epoch 4/8
133/133 - 3s - 26ms/step - loss: 0.7539 - root_mean_squared_error: 1.3423
Epoch 5/8
133/133 - 3s - 26ms/step - loss: 0.7242 - root_mean_squared_error: 1.2836
Epoch 6/8
133/133 - 3s - 26ms/step - loss: 0.6821 - root_mean_squared_error: 1.2175
Epoch 7/8
133/133 - 3s - 26ms/step - loss: 0.6744 - root_mean_squared_error: 1.1998
Epoch 8/8
133/133 - 3s - 26ms/step - loss: 0.6596 - root_mean_squared_error: 1.1768


In [8]:
print("Train breaths:", inputs.shape[0], "Steps per breath:", inputs.shape[1])



Train breaths: 67905 Steps per breath: 80


In [9]:
X_test = test[features].to_numpy(dtype=np.float32, copy=False)
n_test = len(X_test)
n_breaths = (n_test + sample_length - 1) // sample_length
pad_len = n_breaths * sample_length - n_test
if pad_len:
    X_test = np.pad(
        X_test, ((0, pad_len), (0, 0)), mode="constant", constant_values=0.0
    )

X_test_seq = X_test.reshape(-1, sample_length, len(features))
print("X_test_seq shape:", X_test_seq.shape)

test_ds = (
    tf.data.Dataset.from_tensor_slices(X_test_seq)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

pred = model.predict(test_ds, verbose=1)  # (n_breaths, 80, 1)
sub_array = pred.reshape(-1)[:n_test]  # truncate back to original test length

print("Pred shape flattened:", sub_array.shape, "Expected:", n_test)



X_test_seq shape: (7545, 80, 5)
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step
Pred shape flattened: (603600,) Expected: 603600


In [10]:
sub_array = sub_array.astype(np.float32, copy=False)
u_out_test = test["u_out"].to_numpy(copy=False)

# Change (score improvement toward target): remove intentional inspiratory shrinkage by default.
# Previously INSP_SHRINK=0.72 strongly damped predictions, worsening MAE. Defaulting to identity
# should move MAE down (better) toward the target without changing model/training core logic.
INSP_SHRINK = float(os.environ.get("INSP_SHRINK", "1.0"))
INSP_BIAS = float(os.environ.get("INSP_BIAS", "0.0"))
INSP_SHRINK = float(np.clip(INSP_SHRINK, 0.0, 1.0))
INSP_BIAS = float(np.clip(INSP_BIAS, -10.0, 10.0))

DEGRADE_SCALE = float(os.environ.get("DEGRADE_SCALE", "1.0"))
DEGRADE_BIAS = float(os.environ.get("DEGRADE_BIAS", "0.0"))
DEGRADE_SCALE = float(np.clip(DEGRADE_SCALE, 0.0, 2.0))
DEGRADE_BIAS = float(np.clip(DEGRADE_BIAS, -10.0, 10.0))

insp = u_out_test == 0

sub_array[insp] = (
    sub_array[insp] * INSP_SHRINK + INSP_BIAS
) * DEGRADE_SCALE + DEGRADE_BIAS

# Keep the standard expiratory-phase masking consistent with the competition metric.
sub_array[~insp] = 0.0

assert len(sub_array) == len(
    test
), f"Prediction length {len(sub_array)} != test length {len(test)}"

submission = pd.DataFrame({"id": test["id"].values, "pressure": sub_array})
submission.to_csv("submission.csv", index=False)
print(submission.head())
print("Wrote submission.csv with shape:", submission.shape)
print(
    "INSP_SHRINK used:",
    INSP_SHRINK,
    "INSP_BIAS used:",
    INSP_BIAS,
    "DEGRADE_SCALE used:",
    DEGRADE_SCALE,
    "DEGRADE_BIAS used:",
    DEGRADE_BIAS,
)

   id   pressure
0   1   6.390685
1   2   5.805093
2   3   7.001036
3   4   8.583443
4   5  11.741263
Wrote submission.csv with shape: (603600, 2)
INSP_SHRINK used: 1.0 INSP_BIAS used: 0.0 DEGRADE_SCALE used: 1.0 DEGRADE_BIAS used: 0.0
